# Graph Inspection Notebook

This notebook loads and inspects the knowledge graph using the utility functions from `cli/commands/evals/utils.py`.

In [ ]:
from pathlib import Path
import sys
import polars as pl

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from cli.commands.evals.utils import (
    load_graph,
    load_node_metadata,
    compute_pagerank,
    pagerank_to_dataframe,
)

[03/14/26 22:45:48] INFO     Using 'conf/logging.yml' as logging configuration. You can change this ]8;id=277110;file:///Users/ayush/Documents/Zitnik_Lab/optimus/.venv/lib/python3.13/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=933876;file:///Users/ayush/Documents/Zitnik_Lab/optimus/.venv/lib/python3.13/site-packages/kedro/framework/project/__init__.py#270\270]8;;\
                             by setting the KEDRO_LOGGING_CONFIG environment variable accordingly.                 

## Load Graph

In [2]:
nodes_path = Path("data/gold/kg/parquet/nodes.parquet")
edges_path = Path("data/gold/kg/parquet/edges.parquet")

G, edge_type_lookup, node_types, edge_types = load_graph(nodes_path, edges_path)

[03/15/26 03:59:48] INFO     Loaded 192123 nodes with 10 node types: ['ANA', 'BPO', 'CCO', 'DIS',       ]8;id=638440;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=83557;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#42\42]8;;\
                             'DRG', 'EXP', 'GEN', 'MFN', 'PHE', 'PWY']                                             

[03/15/26 04:00:24] INFO     Loaded 21851915 edges with 26 edge types: ['ANA-ANA', 'ANA-PRO',           ]8;id=782955;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=691447;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#61\61]8;;\
                             'BPO-BPO', 'BPO-PRO', 'CCO-CCO', 'CCO-PRO', 'DIS-DIS', 'DIS-PHE',                     
                             'DIS-PRO', 'DRG-DIS', 'DRG-DRG', 'DRG-PHE', 'DRG-PRO', 'EXP-BPO',                     
                             'EXP-CCO', 'EXP-DIS', 'EXP-EXP', 'EXP-MFN', 'EXP-PRO', 'MFN-MFN',                     
                             'MFN-PRO', 'PHE-PHE', 'PHE-PRO', 'PRO-PRO', 'PWY-PRO', 'PWY-PWY']                     

## Basic Statistics

In [3]:
print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")
print(f"\nNode types ({len(node_types)}): {node_types}")
print(f"\nEdge types ({len(edge_types)}): {edge_types}")

Nodes: 192,123
Edges: 21,851,915

Node types (10): ['ANA', 'BPO', 'CCO', 'DIS', 'DRG', 'EXP', 'GEN', 'MFN', 'PHE', 'PWY']

Edge types (26): ['ANA-ANA', 'ANA-PRO', 'BPO-BPO', 'BPO-PRO', 'CCO-CCO', 'CCO-PRO', 'DIS-DIS', 'DIS-PHE', 'DIS-PRO', 'DRG-DIS', 'DRG-DRG', 'DRG-PHE', 'DRG-PRO', 'EXP-BPO', 'EXP-CCO', 'EXP-DIS', 'EXP-EXP', 'EXP-MFN', 'EXP-PRO', 'MFN-MFN', 'MFN-PRO', 'PHE-PHE', 'PHE-PRO', 'PRO-PRO', 'PWY-PRO', 'PWY-PWY']


## Load Node Metadata

In [4]:
node_metadata = load_node_metadata(nodes_path)
node_metadata.head(10)

[03/15/26 04:00:28] INFO     Loaded metadata for 192682 nodes                                           ]8;id=145951;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=474581;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#98\98]8;;\

id,label,name
str,str,str
"""ENSG00000000003""","""GEN""","""tetraspanin 6"""
"""ENSG00000000005""","""GEN""","""tenomodulin"""
"""ENSG00000000419""","""GEN""","""dolichyl-phosphate mannosyltra…"
"""ENSG00000000457""","""GEN""","""SCY1 like pseudokinase 3"""
"""ENSG00000000460""","""GEN""","""FIGNL1 interacting regulator o…"
"""ENSG00000000938""","""GEN""","""FGR proto-oncogene, Src family…"
"""ENSG00000000971""","""GEN""","""complement factor H"""
"""ENSG00000001036""","""GEN""","""alpha-L-fucosidase 2"""
"""ENSG00000001084""","""GEN""","""glutamate-cysteine ligase cata…"


In [ ]:
# Node counts by type
node_metadata.group_by("label").len().sort("len", descending=True)

# Node counts by type from 

label,len
str,u32
"""GEN""",61306
"""DIS""",36992
"""BPO""",25754
"""PHE""",19341
"""DRG""",16766
"""ANA""",14624
"""MFN""",10161
"""CCO""",4052
"""PWY""",2805


## Debug Difference

In [8]:
nodes_path = Path("data/gold/kg/parquet/nodes.parquet")
df = pl.read_parquet(nodes_path)

print("rows:", df.height)
print("unique ids:", df.select(pl.col("id").n_unique()).item())
print("duplicate rows by id:", df.height - df.select(pl.col("id").n_unique()).item())

rows: 192682
unique ids: 192123
duplicate rows by id: 559


In [12]:
# Identify IDs that appear more than once
duplicate_ids = (
    df.group_by("id")
    .len()
    .filter(pl.col("len") > 1)
    .select("id")
)

# Join back to original data to get full metadata for those IDs
duplicated_nodes_table = (
    df.join(duplicate_ids, on="id", how="inner")
    .with_columns(
        pl.col("properties").str.json_path_match("$.name").alias("name")
    )
    .select(["id", "name", "label", "properties"])
    .sort("id")
)

# Save the table
output_file = Path("data/gold/kg/duplicated_nodes.csv")
duplicated_nodes_table.write_csv(output_file)

print(f"Found {duplicate_ids.height} unique duplicated IDs resulting in {duplicated_nodes_table.height} total rows.")
print(f"Table saved to: {output_file}")

# Display the first few rows for inspection
duplicated_nodes_table.head(10)

Found 559 unique duplicated IDs resulting in 1118 total rows.
Table saved to: data/gold/kg/duplicated_nodes.csv


id,name,label,properties
str,str,str,str
"""GO_0000002""","""mitochondrial genome maintenan…","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0000002""","""obsolete mitochondrial genome …","""BPO""","""{""sources"":{""direct"":[""GO""],""i…"
"""GO_0000050""","""urea cycle""","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0000050""","""urea cycle""","""BPO""","""{""sources"":{""direct"":[""GO""],""i…"
"""GO_0000096""","""sulfur amino acid metabolic pr…","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0000096""","""sulfur amino acid metabolic pr…","""BPO""","""{""sources"":{""direct"":[""GO""],""i…"
"""GO_0000226""","""microtubule cytoskeleton organ…","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0000226""","""microtubule cytoskeleton organ…","""BPO""","""{""sources"":{""direct"":[""GO""],""i…"
"""GO_0000278""","""mitotic cell cycle""","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"


In [9]:
dups = (
    df.group_by("id")
    .len()
    .filter(pl.col("len") > 1)
    .sort("len", descending=True)
)

print(dups.head(20))
print("number of duplicated ids:", dups.height)
print("extra duplicate rows:", dups["len"].sum() - dups.height)

shape: (20, 2)
┌────────────┬─────┐
│ id         ┆ len │
│ ---        ┆ --- │
│ str        ┆ u32 │
╞════════════╪═════╡
│ GO_0071696 ┆ 2   │
│ GO_0006996 ┆ 2   │
│ GO_0006517 ┆ 2   │
│ GO_0044403 ┆ 2   │
│ GO_0019395 ┆ 2   │
│ …          ┆ …   │
│ GO_0006541 ┆ 2   │
│ GO_0009058 ┆ 2   │
│ HP_0002619 ┆ 2   │
│ GO_0045839 ┆ 2   │
│ GO_0010960 ┆ 2   │
└────────────┴─────┘
number of duplicated ids: 559
extra duplicate rows: 559


In [ ]:
dup_ids = dups["id"].head(10).to_list()
df.filter(pl.col("id").is_in(dup_ids)).sort(["id", "label"])

id,label,properties
str,str,str
"""GO_0005977""","""BPO""","""{""sources"":{""direct"":[""GO""],""i…"
"""GO_0005977""","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0006517""","""BPO""","""{""sources"":{""direct"":[""GO""],""i…"
"""GO_0006517""","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0006996""","""BPO""","""{""sources"":{""direct"":[""GO""],""i…"
…,…,…
"""GO_0044403""","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0046877""","""BPO""","""{""sources"":{""direct"":[""GO""],""i…"
"""GO_0046877""","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"
